In [ ]:
# Data Clean and Prepocessing
from pathlib import Path
import csv
import random
from collections import Counter, defaultdict

PROJECT_ROOT = Path.cwd()

HAM_DIR = PROJECT_ROOT / "data" / "ham10000"
TEST_DIR = PROJECT_ROOT / "data" / "test_official"
OUTPUT_SPLIT_DIR = PROJECT_ROOT / "data" / "splits"

HAM_METADATA = HAM_DIR / "HAM10000_metadata.csv"
HAM_IMAGE_DIRS = [
    HAM_DIR / "HAM10000_images_part_1",
    HAM_DIR / "HAM10000_images_part_2",
]

TEST_METADATA = TEST_DIR / "ISIC2018_Task3_Test_GroundTruth.csv"
TEST_IMAGE_DIR = TEST_DIR / "ISIC2018_Task3_Test_Images"

LABEL_ORDER = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
OUTPUT_FIELDS = [
    "lesion_id",
    "image_id",
    "dx",
    "dx_type",
    "age",
    "sex",
    "localization",
    "image_path",
    "split",
    "dataset_source",
    "source_dataset",
]

def read_csv(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    with path.open(newline="", encoding="utf-8-sig") as file:
        return list(csv.DictReader(file))

def write_csv(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=OUTPUT_FIELDS)
        writer.writeheader()
        writer.writerows(rows)

def relative_path(path):
    return path.relative_to(PROJECT_ROOT).as_posix()

def build_image_map(image_dirs):
    image_map = {}
    for image_dir in image_dirs:
        if not image_dir.exists():
            raise FileNotFoundError(f"Missing image folder: {image_dir}")
        for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
            for path in image_dir.glob(ext):
                image_map[path.stem] = relative_path(path)
    return image_map

def normalize_label(label):
    label = str(label).strip().lower()
    if label == "ak":
        return "akiec"
    return label

def label_from_test_row(row):
    if row.get("dx"):
        return normalize_label(row["dx"])
    for label in LABEL_ORDER:
        for key in [label, label.upper()]:
            if key in row and str(row[key]).strip() in {"1", "1.0"}:
                return label
    raise ValueError(f"Cannot find label for test row: {row}")

def make_ham_rows():
    ham_map = build_image_map(HAM_IMAGE_DIRS)
    metadata = read_csv(HAM_METADATA)
    rows = []
    missing = []
    for row in metadata:
        image_id = row["image_id"]
        if image_id not in ham_map:
            missing.append(image_id)
            continue
        rows.append({
            "lesion_id": row.get("lesion_id", ""),
            "image_id": image_id,
            "dx": normalize_label(row.get("dx", "")),
            "dx_type": row.get("dx_type", ""),
            "age": row.get("age", ""),
            "sex": row.get("sex", ""),
            "localization": row.get("localization", ""),
            "image_path": ham_map[image_id],
            "split": "",
            "dataset_source": "HAM10000_train_val",
            "source_dataset": "",
        })
    if missing:
        raise FileNotFoundError(f"Missing HAM10000 images: {len(missing)}. Example: {missing[:5]}")
    return rows

def stratified_train_val_split(rows, val_frac=0.15, seed=42):
    rng = random.Random(seed)
    by_label = defaultdict(list)
    for row in rows:
        by_label[row["dx"]].append(row)
    train_rows = []
    val_rows = []
    for label in LABEL_ORDER:
        group = list(by_label[label])
        rng.shuffle(group)
        val_n = round(len(group) * val_frac)
        val_part = group[:val_n]
        train_part = group[val_n:]
        for row in train_part:
            item = dict(row)
            item["split"] = "train"
            train_rows.append(item)
        for row in val_part:
            item = dict(row)
            item["split"] = "val"
            val_rows.append(item)
    return train_rows, val_rows

def make_test_rows():
    test_map = build_image_map([TEST_IMAGE_DIR])
    metadata = read_csv(TEST_METADATA)
    rows = []
    missing = []
    for index, row in enumerate(metadata):
        image_id = row.get("image_id") or row.get("image") or row.get("ID") or row.get("id")
        if image_id is None:
            raise ValueError("The test CSV must contain image_id, image, ID, or id column.")
        image_id = image_id.replace(".jpg", "").replace(".jpeg", "").replace(".png", "")
        if image_id not in test_map:
            missing.append(image_id)
            continue
        rows.append({
            "lesion_id": row.get("lesion_id", f"HAMTEST_{index:07d}"),
            "image_id": image_id,
            "dx": label_from_test_row(row),
            "dx_type": row.get("dx_type", ""),
            "age": row.get("age", ""),
            "sex": row.get("sex", ""),
            "localization": row.get("localization", ""),
            "image_path": test_map[image_id],
            "split": "test",
            "dataset_source": "school_test",
            "source_dataset": row.get("dataset", ""),
        })
    if missing:
        print(f"Warning: excluded {len(missing)} test rows because image files were missing.")
        print("Example missing test image IDs:", missing[:5])
    return rows

def show_counts(name, rows):
    counts = Counter(row["dx"] for row in rows)
    print(name, len(rows), {label: counts[label] for label in LABEL_ORDER})
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Checking input files and folders")
for path in [HAM_METADATA, *HAM_IMAGE_DIRS, TEST_METADATA, TEST_IMAGE_DIR]:
    print(path, "OK" if path.exists() else "MISSING")

ham_rows = make_ham_rows()
train_rows, val_rows = stratified_train_val_split(ham_rows, val_frac=0.15, seed=42)
test_rows = make_test_rows()

write_csv(OUTPUT_SPLIT_DIR / "train_seed42.csv", train_rows)
write_csv(OUTPUT_SPLIT_DIR / "val_seed42.csv", val_rows)
write_csv(OUTPUT_SPLIT_DIR / "test_school.csv", test_rows)

show_counts("train", train_rows)
show_counts("val", val_rows)
show_counts("test", test_rows)

print("DONE")
print(OUTPUT_SPLIT_DIR / "train_seed42.csv")
print(OUTPUT_SPLIT_DIR / "val_seed42.csv")
print(OUTPUT_SPLIT_DIR / "test_school.csv")

def compute_class_weights(rows):
    counts = Counter(row["dx"] for row in rows)
    total = sum(counts[label] for label in LABEL_ORDER)
    num_classes = len(LABEL_ORDER)
    return {label: total / (num_classes * counts[label]) for label in LABEL_ORDER}

class_weight_dict = compute_class_weights(train_rows)
class_weight_list = [class_weight_dict[label] for label in LABEL_ORDER]
print("Class weights:")
for label, weight in class_weight_dict.items():
    print(label, round(weight, 3))

try:
    import torch
    class_weights_tensor = torch.tensor(class_weight_list, dtype=torch.float32)
    print("class_weights_tensor:", class_weights_tensor)
except Exception as error:
    class_weights_tensor = None
    print("Torch is not available. Use class_weight_list instead.")
    print("Import error:", error)

import numpy as np
from PIL import Image

def resolve_image_path(path_value):
    path = Path(str(path_value))
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path

def compute_channel_mean_std(rows, image_size=(128, 128)):
    channel_sum = np.zeros(3, dtype=np.float64)
    channel_sq_sum = np.zeros(3, dtype=np.float64)
    pixel_count = 0
    for row in rows:
        image = Image.open(resolve_image_path(row["image_path"])).convert("RGB")
        image = image.resize(image_size)
        array = np.asarray(image, dtype=np.float32) / 255.0
        channel_sum += array.sum(axis=(0, 1))
        channel_sq_sum += (array ** 2).sum(axis=(0, 1))
        pixel_count += array.shape[0] * array.shape[1]
    mean = channel_sum / pixel_count
    std = np.sqrt(channel_sq_sum / pixel_count - mean ** 2)
    return mean.tolist(), std.tolist()

CNN_TRAIN_MEAN, CNN_TRAIN_STD = compute_channel_mean_std(train_rows, image_size=(128, 128))
print("CNN train mean:", [round(value, 4) for value in CNN_TRAIN_MEAN])
print("CNN train std:", [round(value, 4) for value in CNN_TRAIN_STD])



In [ ]:
try:
    from torchvision import transforms
except Exception as error:
    transforms = None
    print("Torchvision is not available. CNN transform objects were not created.")
    print("Import error:", error)

if transforms is not None:
    cnn_baseline_train_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
    ])

    cnn_baseline_eval_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
    ])

    cnn_advanced_train_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=45),
        transforms.ToTensor(),
        transforms.Normalize(mean=CNN_TRAIN_MEAN, std=CNN_TRAIN_STD),
    ])

    cnn_advanced_eval_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=CNN_TRAIN_MEAN, std=CNN_TRAIN_STD),
    ])

    print("CNN transforms created.")
else:
    cnn_baseline_train_transform = None
    cnn_baseline_eval_transform = None
    cnn_advanced_train_transform = None
    cnn_advanced_eval_transform = None

import pandas as pd
from PIL import Image, ImageDraw, ImageFont

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    from torchvision import transforms as tv_transforms
except Exception:
    tv_transforms = None

def pil_random_augmentation(image):
    image = image.resize((128, 128))
    if random.random() < 0.5:
        image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    angle = random.uniform(-45, 45)
    return image.rotate(angle, resample=Image.Resampling.BILINEAR, fillcolor=(0, 0, 0))

train_df = pd.read_csv(OUTPUT_SPLIT_DIR / "train_seed42.csv")
sample_img_path = resolve_image_path(train_df.iloc[0]["image_path"])

raw_img = Image.open(sample_img_path).convert("RGB")
orig_img = raw_img.resize((128, 128))

if tv_transforms is not None:
    vis_transform = tv_transforms.Compose([
        tv_transforms.Resize((128, 128)),
        tv_transforms.RandomHorizontalFlip(p=0.5),
        tv_transforms.RandomRotation(degrees=45),
    ])
    augmented_imgs = [vis_transform(orig_img) for _ in range(3)]
else:
    augmented_imgs = [pil_random_augmentation(orig_img) for _ in range(3)]

output_path = PROJECT_ROOT / "data_augmentation.png"

if plt is not None:
    fig, axes = plt.subplots(1, 4, figsize=(15, 4))
    axes[0].imshow(orig_img)
    axes[0].set_title("Original (128x128)", fontsize=11, pad=10)
    axes[0].axis("off")
    for index, image in enumerate(augmented_imgs):
        axes[index + 1].imshow(image)
        axes[index + 1].set_title(f"Augmented Variant {index + 1}", fontsize=11, pad=10)
        axes[index + 1].axis("off")
    plt.suptitle(
        "Figure 4.2: Comparison Between Raw Dermoscopic Image and Stochastic Augmentations",
        fontsize=13,
        y=0.02,
    )
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
else:
    tile_w, tile_h = 300, 300
    title_h = 58
    caption_h = 52
    gap = 18
    canvas_w = tile_w * 4 + gap * 5
    canvas_h = title_h + tile_h + caption_h + 18
    canvas = Image.new("RGB", (canvas_w, canvas_h), "white")
    draw = ImageDraw.Draw(canvas)
    try:
        font_title = ImageFont.truetype("Arial.ttf", 22)
        font_caption = ImageFont.truetype("Arial.ttf", 18)
    except Exception:
        font_title = ImageFont.load_default()
        font_caption = ImageFont.load_default()
    images = [orig_img, *augmented_imgs]
    titles = ["Original (128x128)", "Augmented Variant 1", "Augmented Variant 2", "Augmented Variant 3"]
    for index, image in enumerate(images):
        x = gap + index * (tile_w + gap)
        title = titles[index]
        box = draw.textbbox((0, 0), title, font=font_title)
        draw.text((x + (tile_w - (box[2] - box[0])) / 2, 14), title, fill="black", font=font_title)
        canvas.paste(image.resize((tile_w, tile_h)), (x, title_h))
    caption = "Figure 4.2: Comparison Between Raw Dermoscopic Image and Stochastic Augmentations"
    box = draw.textbbox((0, 0), caption, font=font_caption)
    draw.text(((canvas_w - (box[2] - box[0])) / 2, title_h + tile_h + 16), caption, fill="black", font=font_caption)
    canvas.save(output_path)

print("Saved augmentation figure:", output_path)



In [ ]:
# Baseline CNN 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# Set random seed for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# ==========================================
# 1. Device and Label Configuration
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define official diagnostic label mapping
LABEL_ORDER = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
label_map = {label: i for i, label in enumerate(LABEL_ORDER)}

# ==========================================
# 2. Data Loading & Preprocessing
# ==========================================
class SkinDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        """
        Custom Dataset for loading skin lesion images.
        """
        self.df = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        img = Image.open(img_path).convert('RGB')
        label = label_map[row['dx']]
        
        if self.transform:
            img = self.transform(img)
            
        return img, label

# Data preprocessing: resize and transform to tensor
tfms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# Initialize datasets
train_dataset = SkinDataset('data/splits/train_seed42.csv', transform=tfms)
val_dataset = SkinDataset('data/splits/val_seed42.csv', transform=tfms)
test_dataset = SkinDataset('data/splits/test_school.csv', transform=tfms)

# Initialize data loaders for training, validation, and testing
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# ==========================================
# 3. Model Building (Baseline CNN)
# ==========================================
class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolutional base
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # Linear classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 7)
        )
        
    def forward(self, x):
        return self.classifier(self.features(x))

model = BaselineCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ==========================================
# 4. Training, Validation and Evaluation Step Definitions
# ==========================================
def train_epoch(dataloader, model, criterion, optimizer):
    model.train()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = criterion(pred, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        all_preds.extend(pred.argmax(1).cpu().numpy())
        all_targets.extend(y.cpu().numpy())
        
    avg_loss = total_loss / len(dataloader)
    all_targets = np.array(all_targets)
    all_preds = np.array(all_preds)
    
    # Calculate performance metrics
    acc = accuracy_score(all_targets, all_preds)
    _, _, train_f1, _ = precision_recall_fscore_support(
        all_targets, all_preds, average='macro', zero_division=0
    )
    
    print("\n--- [Train Stage] Results ---")
    print(f"Train Loss: {avg_loss:.4f} | Train Accuracy: {acc*100:.2f}% | Train Macro-F1: {train_f1:.4f}")
    return avg_loss, acc, train_f1

def evaluate(dataloader, model, criterion, desc="Validation"):
    """
    Evaluates the model on validation or test sets.
    """
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = criterion(pred, y)
            total_loss += loss.item()
            
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_targets.extend(y.cpu().numpy())
            
    avg_loss = total_loss / len(dataloader)
    all_targets = np.array(all_targets)
    all_preds = np.array(all_preds)
    
    # Calculate performance metrics
    acc = accuracy_score(all_targets, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_targets, all_preds, average='macro', zero_division=0
    )
    
    print(f"\n--- [{desc} Stage] Results ---")
    print(f"Loss: {avg_loss:.4f} | Accuracy: {acc*100:.2f}%")
    print(f"Macro Precision: {precision:.4f} | Macro Recall: {recall:.4f} | Macro F1-Score: {f1:.4f}")
    
    # Generate confusion matrix
    cm = confusion_matrix(all_targets, all_preds)
    print("Confusion Matrix:")
    print(cm)
    
    return avg_loss, acc, f1, cm

# ==========================================
# 5. Training and Validation Process
# ==========================================
epochs = 15
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []
train_f1s, val_f1s = [], []

for epoch in range(epochs):
    print(f"\n==================== Epoch {epoch+1}/{epochs} ====================")
    
    # Execute training phase
    train_loss, train_acc, train_f1 = train_epoch(train_loader, model, criterion, optimizer)
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    train_f1s.append(train_f1)
    
    # Execute validation phase
    val_loss, val_acc, val_f1, _ = evaluate(val_loader, model, criterion, desc="Validation")
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_f1s.append(val_f1)

# ==========================================
# 6. Model Saving
# ==========================================
# Save the final model weights
torch.save(model.state_dict(), 'baseline_cnn_model.pth')
print("\n[System] Model training completed and weights saved to 'baseline_cnn_model.pth'")

# ==========================================
# 7. Final Test Evaluation
# ==========================================
print("\n" + "="*60)
print("Running Final Evaluation on the Official Test Set...")
print("="*60)

test_loss, test_acc, test_f1, test_cm = evaluate(test_loader, model, criterion, desc="Official Test")

# ==========================================
# 8. Visualization (Metrics Curves & Confusion Matrix)
# ==========================================
epochs_x = np.arange(epochs)

# Figure 1: Loss, Accuracy, and Macro-F1 side-by-side
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

# Subplot 1: Loss curves
ax1.plot(epochs_x, train_losses, label="Train Loss", color="#1f77b4")
ax1.plot(epochs_x, val_losses, label="Val Loss", color="#ff7f0e")
ax1.set_title("Loss", fontsize=14)
ax1.set_xlabel("Epochs", fontsize=11)
ax1.set_xticks(np.arange(0, epochs, 2))
ax1.legend(loc="upper right")

# Subplot 2: Accuracy curves
ax2.plot(epochs_x, train_accuracies, label="Train Acc", color="#1f77b4")
ax2.plot(epochs_x, val_accuracies, label="Val Acc", color="#ff7f0e")
ax2.set_title("Accuracy", fontsize=14)
ax2.set_xlabel("Epochs", fontsize=11)
ax2.set_xticks(np.arange(0, epochs, 2))
ax2.legend(loc="lower right")

# Subplot 3: Macro-F1 curves
ax3.plot(epochs_x, train_f1s, label="Train F1", color="#1f77b4")
ax3.plot(epochs_x, val_f1s, label="Val F1", color="#ff7f0e")
ax3.set_title("Macro-F1", fontsize=14)
ax3.set_xlabel("Epochs", fontsize=11)
ax3.set_xticks(np.arange(0, epochs, 2))
ax3.legend(loc="lower right")

plt.tight_layout()
plt.savefig("metrics_curves_fig1.png", dpi=300)
plt.show()

# Figure 2: Confusion Matrix Heatmap for test set
class_labels = ["AKIEC", "BCC", "BKL", "DF", "MEL", "NV", "VASC"]

plt.figure(figsize=(8, 6.5))
sns.heatmap(
    test_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_labels,
    yticklabels=class_labels,
    cbar=True
)

plt.title("Confusion Matrix (Baseline)", fontsize=14, pad=10)
plt.xlabel("Predicted", fontsize=11)
plt.ylabel("True", fontsize=11)

plt.tight_layout()
plt.savefig("confusion_matrix_fig2.png", dpi=300)
plt.show()

In [ ]:
# CNN Advanced
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# ==========================================
# 0. Reproducibility & Device Configuration
# ==========================================
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

LABEL_ORDER = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
label_map = {label: i for i, label in enumerate(LABEL_ORDER)}

# ==========================================
# 1. Dataset & Preparation
# ==========================================
train_df = pd.read_csv('data/splits/train_seed42.csv')
class_counts = train_df['dx'].value_counts()
counts_list = np.array([class_counts.get(lbl, 0) for lbl in LABEL_ORDER], dtype=np.float32)
class_weights = torch.FloatTensor(len(train_df) / (len(LABEL_ORDER) * counts_list)).to(device)

train_tfms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=45),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_tfms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class SkinDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.df = pd.read_csv(csv_file)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        return self.transform(img), label_map[row['dx']]

train_loader = DataLoader(SkinDataset('data/splits/train_seed42.csv', transform=train_tfms), batch_size=64, shuffle=True, drop_last=True)
val_loader = DataLoader(SkinDataset('data/splits/val_seed42.csv', transform=val_tfms), batch_size=64, shuffle=False)
test_loader = DataLoader(SkinDataset('data/splits/test_school.csv', transform=val_tfms), batch_size=64, shuffle=False)

# ==========================================
# 2. Model Architecture
# ==========================================
class AdvancedCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.3)
        )
        self.avgpool = nn.AdaptiveAvgPool2d((4, 4))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.avgpool(self.features(x)))

model = AdvancedCNN().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# ==========================================
# 3. Helper Functions with Tracking
# ==========================================
def train_epoch(dataloader, model, criterion, optimizer):
    model.train()
    total_loss, all_preds, all_targets = 0, [], []
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        all_preds.extend(pred.argmax(1).cpu().numpy())
        all_targets.extend(y.cpu().numpy())
    
    # Calculate training F1
    _, _, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='macro', zero_division=0)
    return total_loss / len(dataloader), accuracy_score(all_targets, all_preds), f1

def evaluate(dataloader, model, criterion, desc="Validation"):
    model.eval()
    total_loss, all_preds, all_targets = 0, [], []
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = criterion(pred, y)
            total_loss += loss.item()
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_targets.extend(y.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_targets, all_preds)
    p, r, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, f1, confusion_matrix(all_targets, all_preds)

# ==========================================
# 4. Training Loop & Data Collection
# ==========================================
epochs = 15
history = {'train_loss':[], 'train_acc':[], 'train_f1':[], 'val_loss':[], 'val_acc':[], 'val_f1':[]}

for epoch in range(epochs):
    tr_loss, tr_acc, tr_f1 = train_epoch(train_loader, model, criterion, optimizer)
    val_loss, val_acc, val_f1, _ = evaluate(val_loader, model, criterion)
    
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['train_f1'].append(tr_f1)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    print(f"Epoch {epoch+1}/{epochs} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
    if val_f1 == max(history['val_f1']): torch.save(model.state_dict(), 'best_model.pth')

# ==========================================
# 5. Plotting Results
# ==========================================
# Global plotting settings
plt.rcParams['font.sans-serif'] = ['Arial', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

epoch_range = np.arange(1, epochs + 1)
fig1, axes1 = plt.subplots(1, 3, figsize=(18, 5))

# Plot Losses
axes1[0].plot(epoch_range, history['train_loss'], label='Train Loss', color='#1f77b4', linewidth=1.8)
axes1[0].plot(epoch_range, history['val_loss'], label='Val Loss', color='#ff7f0e', linewidth=1.8)
axes1[0].set_title('Loss', fontsize=14, pad=10)
axes1[0].set_xlabel('Epochs', fontsize=11)
axes1[0].legend(loc='upper right')

# Plot Accuracies
axes1[1].plot(epoch_range, history['train_acc'], label='Train Acc', color='#1f77b4', linewidth=1.8)
axes1[1].plot(epoch_range, history['val_acc'], label='Val Acc', color='#ff7f0e', linewidth=1.8)
axes1[1].set_title('Accuracy', fontsize=14, pad=10)
axes1[1].set_xlabel('Epochs', fontsize=11)
axes1[1].legend(loc='lower right')

# Plot F1-Scores
axes1[2].plot(epoch_range, history['train_f1'], label='Train F1', color='#1f77b4', linewidth=1.8)
axes1[2].plot(epoch_range, history['val_f1'], label='Val F1', color='#ff7f0e', linewidth=1.8)
axes1[2].set_title('Macro-F1', fontsize=14, pad=10)
axes1[2].set_xlabel('Epochs', fontsize=11)
axes1[2].legend(loc='lower right')

plt.tight_layout()
plt.show()

# ==========================================
# 6. Test Stage Evaluation & Final Plot
# ==========================================
best_model = AdvancedCNN().to(device)
best_model.load_state_dict(torch.load('best_model.pth', map_location=device))
_, _, _, cm_test = evaluate(test_loader, best_model, criterion, desc="Test")

# Plot Confusion Matrix
plt.figure(figsize=(8, 6.5))
sns.heatmap(data=cm_test, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[l.upper() for l in LABEL_ORDER], 
            yticklabels=[l.upper() for l in LABEL_ORDER])
plt.title('Confusion Matrix (Official Test Stage)', fontsize=14, pad=15)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.show()